In [289]:
import pandas as pd
import ast
import re
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MultiLabelBinarizer
from sklearn.compose import ColumnTransformer 
from sklearn.base import BaseEstimator, TransformerMixin

In [290]:
df = pd.read_csv('updated_aggregated_advertisers_dataset_v2.csv')


### Cleaning data of (unique_goals, order_goals) columns

In [291]:
def clean_messy_list_string(x):
    if pd.isna(x) or x == "":
        return [] 
    
    x = str(x).upper()
    words = re.findall(r'[A-Z_]{3,}', x)
    clean_words = [word.lower() for word in words if word]
    return clean_words  

df['unique_goals'] = df['unique_goals'].apply(clean_messy_list_string)
df['order_goals'] = df['order_goals'].apply(clean_messy_list_string)

### Cleaning data of (postal_codes) for vectorization

In [292]:
df['postal_codes'] = df['postal_codes'].apply(ast.literal_eval)

### checking order_goals and unique_goals rows similarity

In [293]:
# Show which rows are the same
same_rows = df[df['order_goals'] == df['unique_goals']]
len(same_rows)

# Show which rows are different
different_rows = df[~(df['order_goals'] == df['unique_goals'])]
len(different_rows)

percentage_different = (len(different_rows) / len(df)) * 100
percentage_same = (len(same_rows) / len(df)) * 100

print(f"{percentage_different:.2f}% of the rows are different.")
print(f"{percentage_same:.2f}% of the rows are identical.")
print(f"{len(same_rows)} out of {len(df)} rows are identical.")

0.00% of the rows are different.
100.00% of the rows are identical.
1390 out of 1390 rows are identical.


### Next three rows for my understanding data. 

In [294]:
# df[['advertiser_name' ,'unique_goals','unique_industries', 'avg_order_duration_days', 'states', 'cities', 'locations', 'postal_codes']].head()

In [295]:
# df[['unique_goals','unique_industries', 'avg_order_duration_days', 'states', 'cities', 'locations', 'postal_codes', 'advertiser_favorite_postal_code', 'favorite_city', 'favorite_state', 'advertiser_postal_code', 'advertiser_city', 'advertiser_state_code']].head()

In [296]:
# df[['unique_goals','unique_industries', 'states', 'cities', 'favorite_city', 'favorite_state', 'advertiser_city', 'advertiser_state_code']].head()

### BOW Vectorization of columns_to_use

In [297]:
columns_to_use = ['unique_goals', 'unique_industries', 'states', 'cities', 
                  'favorite_city', 'favorite_state', 'advertiser_city', 'advertiser_state_code']

def preprocess_text_preserve_multiword(series):
    return series.astype(str) \
        .str.replace(r"[\[\]']", "", regex=True) \
        .str.replace(r"[&]", "", regex=True) \
        .str.split(",") \
        .apply(lambda items: ' '.join(
            re.sub(r'[^\w\s]', '', item).strip().replace(" ", "_")
            for item in items if item.strip()
        ))

def create_semantic_bow_vectors(df):
    geographic_cols = ['states', 'cities', 'favorite_city', 'favorite_state', 
                      'advertiser_city', 'advertiser_state_code']
    goal_cols = ['unique_goals']
    industry_cols = ['unique_industries']
    
    vectors = {}
    vectorizers = {}
    
    # 1. Geographic BOW Vector
    geo_text = df[geographic_cols].apply(preprocess_text_preserve_multiword)
    geo_text_combined = geo_text.apply(lambda row: ' '.join(row), axis=1)
    geo_vectorizer = CountVectorizer(lowercase=True, max_features=200)
    geo_bow = geo_vectorizer.fit_transform(geo_text_combined)
    geo_features = [f"{name}" for name in geo_vectorizer.get_feature_names_out()]
    vectors['geographic'] = pd.DataFrame(geo_bow.toarray(), columns=geo_features, index=df.index)
    vectorizers['geographic'] = geo_vectorizer
    
    # 2. Goals BOW Vector
    goal_text = df[goal_cols].apply(preprocess_text_preserve_multiword)
    goal_text_combined = goal_text.apply(lambda row: ' '.join(row), axis=1)
    goal_vectorizer = CountVectorizer(lowercase=True, max_features=50)
    goal_bow = goal_vectorizer.fit_transform(goal_text_combined)
    goal_features = [f"{name}" for name in goal_vectorizer.get_feature_names_out()]
    vectors['goals'] = pd.DataFrame(goal_bow.toarray(), columns=goal_features, index=df.index)
    vectorizers['goals'] = goal_vectorizer
    
    # 3. Industry BOW Vector
    industry_text = df[industry_cols].apply(preprocess_text_preserve_multiword)
    industry_text_combined = industry_text.apply(lambda row: ' '.join(row), axis=1)
    industry_vectorizer = CountVectorizer(lowercase=True, max_features=100)
    industry_bow = industry_vectorizer.fit_transform(industry_text_combined)
    industry_features = [f"{name}" for name in industry_vectorizer.get_feature_names_out()]
    vectors['industry'] = pd.DataFrame(industry_bow.toarray(), columns=industry_features, index=df.index)
    vectorizers['industry'] = industry_vectorizer
    
    return vectors, vectorizers

# Create semantic vectors
semantic_vectors, semantic_vectorizers = create_semantic_bow_vectors(df)

print("Geographic BOW shape:", semantic_vectors['geographic'].shape)
print("Goals BOW shape:", semantic_vectors['goals'].shape) 
print("Industry BOW shape:", semantic_vectors['industry'].shape)

# Option 1: Use vectors separately for different analyses
geo_similarity = semantic_vectors['geographic']
goal_similarity = semantic_vectors['goals']
industry_similarity = semantic_vectors['industry']

# Option 2: Combine all vectors (if needed for single model)
combined_semantic_bow = pd.concat([
    semantic_vectors['geographic'],
    semantic_vectors['goals'], 
    semantic_vectors['industry']
], axis=1)

combined_semantic_bow.head()

Geographic BOW shape: (1390, 200)
Goals BOW shape: (1390, 5)
Industry BOW shape: (1390, 30)


,afton,albany,alexandria,altoona,andover,ar,arcadia,arlington,ashland,aurora,...,law__government,news,online_communities,people__society,pets__animals,real_estate,reference,shopping,sports,travel
0,0,2,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### Applying StandardScaler, MultiLabelBinarizerTransformer, OneHotEncoder

In [298]:
class MultiLabelBinarizerTransformer(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.mlb = MultiLabelBinarizer()

    def fit(self, X, y=None):
        return self.mlb.fit(X)

    def transform(self, X):
        return self.mlb.transform(X)

    def get_feature_names_out(self, input_features=None):
        return self.mlb.classes_

postal_code_encoder = MultiLabelBinarizerTransformer()
advertiser_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

preprocessor = ColumnTransformer(transformers=[
    ('duration', StandardScaler(), ['avg_order_duration_days']),
    ('postal_codes', postal_code_encoder, 'postal_codes'),
    ('advertiser_zip', advertiser_encoder, ['advertiser_postal_code'])
], remainder='drop')

X_transformed = preprocessor.fit_transform(df)
duration_features = ['avg_order_duration_days']

postal_code_encoder.fit(df['postal_codes'])
postal_code_features = list(postal_code_encoder.mlb.classes_)

advertiser_features = list(preprocessor.named_transformers_['advertiser_zip'].get_feature_names_out(['advertiser_postal_code']))
all_feature_names = duration_features + postal_code_features + advertiser_features

df_transformed = pd.DataFrame(X_transformed, columns=all_feature_names)
df_transformed.head()

,avg_order_duration_days,00501,00544,01001,01002,01005,01008,01011,01012,01013,...,advertiser_postal_code_98272.0,advertiser_postal_code_98273.0,advertiser_postal_code_98337.0,advertiser_postal_code_98520.0,advertiser_postal_code_98815.0,advertiser_postal_code_98816.0,advertiser_postal_code_98926.0,advertiser_postal_code_99206.0,advertiser_postal_code_99336.0,advertiser_postal_code_99508.0
0,-0.274946,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2.514669,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.468537,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.080150,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1.472657,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### Concatinating both vector matrices 

In [299]:
combined_semantic_bow = combined_semantic_bow.reset_index(drop=True)
df_transformed = df_transformed.reset_index(drop=True)

final_features_df = pd.concat([df_transformed, combined_semantic_bow], axis=1)
final_features_df.head()


,avg_order_duration_days,00501,00544,01001,01002,01005,01008,01011,01012,01013,...,law__government,news,online_communities,people__society,pets__animals,real_estate,reference,shopping,sports,travel
0,-0.274946,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
1,2.514669,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
2,0.468537,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
3,0.080150,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
4,1.472657,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0


# Currently working on this part (NOT COMPLETED YET)

In [300]:
# from scipy.sparse import hstack, csr_matrix
# from sklearn.metrics.pairwise import cosine_similarity
# import pandas as pd

# def recommend_postal_codes(user_input, final_features_df, final_df, preprocessor, semantic_vectorizers):
#     # Step 1: Extract structured, goals, and industry
#     structured_data = pd.DataFrame([{
#         'adv_name': user_input['adv_name'],  # if adv_name is needed
#         'duration_days': user_input['duration']
#     }])
#     user_goals = [user_input['goals']]
#     user_industry = [user_input['industry']]

#     # Step 2: Transform using fitted encoders / vectorizers
#     user_structured = preprocessor.transform(structured_data)
#     goal_vector = semantic_vectorizers['goals'].transform(user_goals)
#     industry_vector = semantic_vectorizers['industry'].transform(user_industry)

#     # Step 3: Convert to sparse if necessary and hstack
#     user_final_vector = hstack([
#         user_structured if not isinstance(user_structured, np.ndarray) else csr_matrix(user_structured),
#         csr_matrix(goal_vector),
#         csr_matrix(industry_vector)
#     ])

#     # Step 4: Ensure final_features_df is sparse too
#     if isinstance(final_features_df, pd.DataFrame):
#         final_features_sparse = csr_matrix(final_features_df.values)
#     else:
#         final_features_sparse = final_features_df  # if already sparse matrix

#     # Step 5: Compute cosine similarity
#     similarity_scores = cosine_similarity(user_final_vector, final_features_sparse)[0]

#     # Step 6: Get top 5 indices
#     top_indices = similarity_scores.argsort()[::-1][:5]

#     # Step 7: Get corresponding postal codes
#     top_postal_codes = final_df.iloc[top_indices]['postal_code'].values

#     return top_postal_codes


In [301]:
# user_input = {
#     'adv_name': 'n fox jewelers',
#     'industry': 'Apparel & Accessories',
#     'goals': 'AWARENESS',
#     'duration': 40.0
# }

# top_postal_codes = recommend_postal_codes(user_input, final_features_df, df, preprocessor, semantic_vectorizers)
# print(top_postal_codes)
